In [5]:
print(df.columns.tolist())

['user_id', 'product_id', 'category', 'subcategory', 'brand', 'price', 'discount', 'final_price', 'rating', 'review_count', 'stock', 'seller_id', 'seller_rating', 'purchase_date', 'shipping_time_days', 'location', 'device', 'payment_method', 'is_returned', 'delivery_status']


In [7]:
import pandas as pd
import numpy as np

# 1. Load your uploaded dataset
df = pd.read_csv('/content/amazon_ecommerce_1M.csv')

# 2. Clean the return column (Using your exact 'is_returned' column)
df['is_returned'] = df['is_returned'].apply(lambda x: 1 if str(x).strip().lower() in ['yes', '1', '1.0', 'true'] else 0)

# 3. Calculate return rates by category (Using lowercase 'category')
category_summary = df.groupby('category').agg(
    Total_Orders=('user_id', 'count'),
    Total_Returns=('is_returned', 'sum')
).reset_index()

category_summary['Return_Rate_%'] = round((category_summary['Total_Returns'] / category_summary['Total_Orders']) * 100, 2)

print("--- Data Analytics Aggregation Summary ---")
print(category_summary.sort_values(by='Return_Rate_%', ascending=False))

--- Data Analytics Aggregation Summary ---
      category  Total_Orders  Total_Returns  Return_Rate_%
0       Beauty        199327          23433          11.76
1     Clothing        199824          23452          11.74
4       Sports        199889          23279          11.65
3         Home        200922          23275          11.58
2  Electronics        200038          22551          11.27


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Prepare Categorical Features (Using lowercase 'category')
df_encoded = pd.get_dummies(df, columns=['category'], drop_first=True)

# 2. Extract Features (X) and Target (y)
# Using the specific numeric columns present in your dataset
feature_cols = ['price', 'discount', 'final_price', 'rating', 'shipping_time_days'] + [col for col in df_encoded.columns if 'category_' in col]
X = df_encoded[feature_cols].fillna(0)
y = df_encoded['is_returned']

# 3. Scale Features and Train Model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(max_iter=1000)
model.fit(X_scaled, y)

# 4. Generate the Return Risk Score for every single order
df['Return_Risk_Score'] = model.predict_proba(X_scaled)[:, 1]
df['Return_Risk_Score'] = df['Return_Risk_Score'].round(2)

# 5. Export the scored data as a brand new CSV file
df.to_csv('ecommerce_with_risk_scores.csv', index=False)
print("Success! 'ecommerce_with_risk_scores.csv' has been generated and saved to your Colab workspace.")

Success! 'ecommerce_with_risk_scores.csv' has been generated and saved to your Colab workspace.
